> **Public reproducibility copy.** Notebook outputs and execution counts were cleared before public packaging to avoid local-path/account leakage. The scientific code and configuration cells are preserved. See the repository README for required data paths and reproducibility scope.

# 01 — RETFound-MAE CFP frozen encoder + PCA128–SVM

**Reviewer point:** R2.2 — retinal foundation/self-supervised baseline.  
**Status before running:** pending.  
**Method label:** `RETFOUND_MAE_CFP_FROZEN_PCA128_SVM`.

This notebook uses the same 594 canonical ROI images and immutable outer folds. The encoder is frozen. The downstream locked protocol is `SMOTE(k=5) → StandardScaler → PCA128 → RBF-SVM(C=10, gamma=scale, class_weight=balanced)`.

## Important: gated RETFound checkpoint

The official checkpoint is gated on Hugging Face. Before running the checkpoint cell, use **one** of these routes:

1. **Preferred local route:** download the approved checkpoint once and place it at `weights/RETFound_mae_natureCFP.pth`.
2. **Authenticated route:** request/accept access on the model page, then authenticate with a read token using `HF_TOKEN`, an existing Hugging Face login, or the interactive login prompt in this notebook.

Never paste a Hugging Face token into a saved notebook cell. The token must not appear in exported outputs, GitHub, or supplementary files.


In [ ]:

# Run once in a fresh environment:
%pip install -r ../requirements-retfound.txt


In [ ]:

from pathlib import Path
import os, sys, json, time, subprocess, hashlib, platform
import numpy as np, pandas as pd, yaml

from pathlib import Path
import sys

def _find_package_root() -> Path:
    marker = Path('config') / 'AMD_R2_2_BASELINE_CONFIG_LOCK_v1.yaml'
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / marker).exists():
            return candidate
    for anchor in [Path('/mnt/data'), Path('/content')]:
        if anchor.exists():
            hits = list(anchor.glob('**/config/AMD_R2_2_BASELINE_CONFIG_LOCK_v1.yaml'))
            if len(hits) == 1:
                return hits[0].parents[1]
    raise FileNotFoundError(
        'Could not locate the extracted AMD_R2_2_CONTEMPORARY_BASELINES_PACKAGE_v1 folder. '
        'Extract the complete ZIP, open the notebook from its notebooks/ folder, and rerun from the top.'
    )

PKG = _find_package_root()
SRC_DIR = PKG / 'src'
if not (SRC_DIR / 'amd_r22_common.py').exists():
    raise FileNotFoundError(
        f'Missing shared module: {SRC_DIR / "amd_r22_common.py"}. '
        'Use package v1.2 or later; v1.1 accidentally omitted the src/ directory.'
    )
sys.path.insert(0, str(SRC_DIR))
from amd_r22_common import *

with open(PKG/'config'/'AMD_R2_2_BASELINE_CONFIG_LOCK_v1.yaml',encoding='utf-8') as f: CFG=yaml.safe_load(f)
RCFG=CFG['retfound']
ROI_ZIP=Path('ROI_594.zip')  # change if needed
ROI_ROOT=extract_zip_if_needed(ROI_ZIP,PKG/'data')
MANIFEST=pd.read_csv(PKG/'manifests/AMD_DATASET_594_R2_2_MANIFEST.csv')
MANIFEST=resolve_manifest_paths(MANIFEST,ROI_ROOT)
preflight=validate_manifest(MANIFEST,ROI_ROOT,CFG['project']['outer_fold_sha256'],PKG/'manifests'/'outer_fold_assignments.csv',True)
assert preflight['complete_and_valid'],preflight
OUTPUT=PKG/'outputs/retfound'; OUTPUT.mkdir(parents=True,exist_ok=True)


## Acquire the official repository and gated checkpoint

This cell first uses a local checkpoint when available. Otherwise it authenticates safely and downloads the gated file. A 401 error normally means either (i) the account has not accepted the model access conditions or (ii) the running environment has no valid token.


In [ ]:
REPO = PKG / 'third_party' / 'RETFound'
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/rmaphoh/RETFound', str(REPO)], check=True)
commit = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
print('RETFound commit:', commit)

CKPT = PKG / 'weights' / 'RETFound_mae_natureCFP.pth'
CKPT.parent.mkdir(parents=True, exist_ok=True)

if CKPT.exists():
    print('Using local RETFound checkpoint:', CKPT)
else:
    from huggingface_hub import hf_hub_download, login, get_token
    try:
        from huggingface_hub.errors import GatedRepoError, HfHubHTTPError
    except ImportError:  # compatibility with older huggingface_hub versions
        from huggingface_hub.utils import GatedRepoError, HfHubHTTPError

    # Priority: environment variable -> previously saved Hugging Face login -> interactive login.
    token = os.environ.get('HF_TOKEN') or get_token()
    if not token:
        print('No Hugging Face token found. An interactive login prompt will open.')
        print('Use a READ token belonging to the same account that has accepted RETFound access.')
        login()
        token = get_token()

    if not token:
        raise RuntimeError(
            'No Hugging Face token is available. First accept access to '
            f"{RCFG['checkpoint_hf_repo']} in your browser, then set HF_TOKEN or run `hf auth login`."
        )

    try:
        downloaded = hf_hub_download(
            repo_id=RCFG['checkpoint_hf_repo'],
            filename=RCFG['checkpoint_filename'],
            token=token,
        )
    except GatedRepoError as exc:
        raise RuntimeError(
            'RETFound access is still gated for the authenticated account. '
            'Open the model page while logged in, accept/share the requested contact information, '
            'wait until access is granted if approval is manual, then rerun this cell. '
            'Alternatively place the approved checkpoint at '
            f'{CKPT}. The token itself must never be stored in the notebook.'
        ) from exc
    except HfHubHTTPError as exc:
        raise RuntimeError(
            'Hugging Face checkpoint download failed. Verify internet access, token validity, '
            'and that the authenticated account has RETFound permission.'
        ) from exc

    import shutil
    shutil.copy2(downloaded, CKPT)

if not CKPT.exists() or CKPT.stat().st_size == 0:
    raise FileNotFoundError(f'RETFound checkpoint is unavailable or empty: {CKPT}')
print('checkpoint:', CKPT, 'size_GiB=', round(CKPT.stat().st_size / 1024**3, 3), 'sha256=', sha256_file(CKPT))


### Authentication troubleshooting

A raw `GatedRepoError: 401` is not an experimental failure. It occurs before model execution. Confirm that the Hugging Face account used by the notebook has accepted the RETFound model conditions. If the environment cannot retain login credentials, download the checkpoint once and use the local route.


In [ ]:
from huggingface_hub import whoami

info = whoami()

print(info)
print("Akun aktif di Jupyter:", info["name"])

In [ ]:
from huggingface_hub import whoami, model_info, hf_hub_download

print("Akun aktif:", whoami()["name"])

model = model_info(
    repo_id="YukunZhou/RETFound_mae_natureCFP",
    token=True,
)

print("Akses metadata berhasil:", model.modelId)

checkpoint_path = hf_hub_download(
    repo_id="YukunZhou/RETFound_mae_natureCFP",
    filename="RETFound_mae_natureCFP.pth",
    token=True,
)

print("Checkpoint tersimpan di:")
print(checkpoint_path)

In [ ]:
from pathlib import Path

CKPT = Path(checkpoint_path).resolve()

assert CKPT.exists(), f"Checkpoint tidak ditemukan: {CKPT}"
assert CKPT.is_file(), f"Path bukan file: {CKPT}"

size_gb = CKPT.stat().st_size / (1024**3)

print("Checkpoint:", CKPT)
print(f"Ukuran: {size_gb:.3f} GiB")
print("Checkpoint ready:", True)

In [ ]:
import torch

checkpoint = torch.load(
    CKPT,
    map_location="cpu",
    weights_only=False
)

print("Checkpoint object type:", type(checkpoint))

if isinstance(checkpoint, dict):
    print("Top-level keys:", checkpoint.keys())

In [ ]:
import gc
import torch

del checkpoint
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Temporary checkpoint object cleared.")

In [ ]:
CKPT = Path(checkpoint_path).resolve()

In [ ]:
from pathlib import Path
import subprocess
import sys

# Folder utama package lokal Anda
import os
PKG = Path(os.environ.get("AMD_R22_PACKAGE_ROOT", Path.cwd())).resolve()

assert PKG.exists(), f"Folder package tidak ditemukan: {PKG}"

# Lokasi repository RETFound
REPO = PKG / "third_party" / "RETFound"
REPO.parent.mkdir(parents=True, exist_ok=True)

# Clone hanya jika repository belum tersedia
if not REPO.exists():
    print("RETFound repository belum ada. Melakukan git clone...")
    subprocess.run(
        [
            "git",
            "clone",
            "https://github.com/rmaphoh/RETFound",
            str(REPO),
        ],
        check=True,
    )

assert REPO.exists(), f"Repository tidak ditemukan: {REPO}"
assert (REPO / "models_vit.py").exists(), (
    f"models_vit.py tidak ditemukan di {REPO}. "
    "Folder repository mungkin tidak lengkap."
)

# Catat commit yang digunakan
commit = subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"],
    text=True,
).strip()

# Masukkan repository ke Python import path
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

import models_vit

print("Package root:", PKG)
print("RETFound repository:", REPO)
print("RETFound commit:", commit)
print("models_vit berhasil di-import")

In [ ]:
import gc
import random
import numpy as np
import torch

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", DEVICE)

# Bangun arsitektur RETFound
model = models_vit.RETFound_mae(
    img_size=224,
    num_classes=3,
    drop_path_rate=0.0,
    global_pool=True,
)

# Muat checkpoint ke CPU terlebih dahulu
checkpoint = torch.load(
    CKPT,
    map_location="cpu",
    weights_only=False,
)

assert isinstance(checkpoint, dict), (
    f"Format checkpoint tidak sesuai: {type(checkpoint)}"
)
assert "model" in checkpoint, (
    f"Key 'model' tidak ditemukan. Keys: {checkpoint.keys()}"
)

state = checkpoint["model"]

# Hapus kemungkinan prefix backbone
state = {
    key.replace("backbone.", ""): value
    for key, value in state.items()
}

# Head pretrained tidak digunakan
state.pop("head.weight", None)
state.pop("head.bias", None)

load_message = model.load_state_dict(
    state,
    strict=False,
)

print("Missing keys:", load_message.missing_keys)
print("Unexpected keys:", load_message.unexpected_keys)

# Ubah keluaran menjadi frozen representation
model.head = torch.nn.Identity()

for parameter in model.parameters():
    parameter.requires_grad = False

model.eval()
model.to(DEVICE)

# Bersihkan objek checkpoint besar dari RAM
del checkpoint
del state
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("RETFound berhasil dimuat.")
print("Expected embedding dimension: 1024")

## Load the frozen RETFound encoder using official model code

In [ ]:

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.transforms import InterpolationMode
from PIL import Image

sys.path.insert(0,str(REPO))
import models_vit

DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
set_seed(42)
model=models_vit.RETFound_mae(img_size=224,num_classes=3,drop_path_rate=0.0,global_pool=True)
checkpoint=torch.load(CKPT,map_location='cpu')
state=checkpoint['model'] if isinstance(checkpoint,dict) and 'model' in checkpoint else checkpoint
state={k.replace('backbone.',''):v for k,v in state.items()}
for k in ['head.weight','head.bias']:
    if k in state: del state[k]
msg=model.load_state_dict(state,strict=False)
print(msg)
model.head=torch.nn.Identity()
model.eval().to(DEVICE)
for p in model.parameters(): p.requires_grad=False

transform=transforms.Compose([
    transforms.Resize(256,interpolation=InterpolationMode.BICUBIC),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485,0.456,0.406),std=(0.229,0.224,0.225)),
])

class ROIDataset(Dataset):
    def __init__(self,df): self.df=df.reset_index(drop=True)
    def __len__(self): return len(self.df)
    def __getitem__(self,i):
        r=self.df.iloc[i]
        with Image.open(r.roi_path) as im: x=transform(im.convert('RGB'))
        return x,r.sample_id


In [ ]:
import sys

SRC_DIR = PKG / "src"

assert (SRC_DIR / "amd_r22_common.py").exists(), (
    f"amd_r22_common.py tidak ditemukan di: {SRC_DIR}"
)

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from amd_r22_common import set_seed

set_seed(42)
print("set_seed berhasil.")

In [ ]:
from amd_r22_common import set_seed

set_seed(42)

print("set_seed berhasil diimpor dan dijalankan.")

In [ ]:
import sys
import gc
import torch

# Import eksplisit karena fungsi ini juga digunakan pada OOF dan LODO
from amd_r22_common import set_seed

# Repository yang sebelumnya sudah di-clone oleh notebook
assert REPO.exists(), f"RETFound repository tidak ditemukan: {REPO}"

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

import models_vit

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

set_seed(42)

# Three-class head hanya diperlukan agar struktur model dapat dibangun.
# Head akan dilepas setelah pretrained backbone dimuat.
model = models_vit.RETFound_mae(
    img_size=224,
    num_classes=3,
    drop_path_rate=0.0,
    global_pool=True,
)

checkpoint = torch.load(
    CKPT,
    map_location="cpu",
    weights_only=False,
)

assert isinstance(checkpoint, dict), type(checkpoint)
assert "model" in checkpoint, checkpoint.keys()

state = checkpoint["model"]

# Defensive cleanup jika checkpoint memiliki prefix backbone.
state = {
    key.replace("backbone.", ""): value
    for key, value in state.items()
}

# Classification head pretrained tidak digunakan.
for key in ("head.weight", "head.bias"):
    if key in state:
        del state[key]

load_message = model.load_state_dict(
    state,
    strict=False,
)

print("Missing keys:", load_message.missing_keys)
print("Unexpected keys:", load_message.unexpected_keys)

# Remove classification head; output becomes the frozen 1024D representation.
model.head = torch.nn.Identity()

for parameter in model.parameters():
    parameter.requires_grad = False

model.eval()
model.to(DEVICE)

# Release the large checkpoint dictionary after weights are copied into model.
del checkpoint
del state
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("RETFound model loaded successfully.")
print("Expected embedding dimension: 1024")

## Extract and audit the 1,024-dimensional feature cache

In [ ]:
from pathlib import Path

# Folder utama package lokal
import os
PKG = Path(os.environ.get("AMD_R22_PACKAGE_ROOT", Path.cwd())).resolve()

assert PKG.exists(), f"Package tidak ditemukan: {PKG}"

# Folder output khusus RETFound
OUTPUT = PKG / "outputs" / "retfound"
OUTPUT.mkdir(parents=True, exist_ok=True)

print("Package root :", PKG)
print("Output folder:", OUTPUT)
print("Output ready :", OUTPUT.exists())

In [ ]:
CACHE_NPZ = OUTPUT / "retfound_mae_cfp_features_594.npz"
CACHE_META = OUTPUT / "retfound_mae_cfp_feature_metadata.csv"

print("Feature cache:", CACHE_NPZ)
print("Metadata     :", CACHE_META)

In [ ]:
from pathlib import Path
import sys
import yaml
import pandas as pd

# ============================================================
# 1. TENTUKAN FOLDER UTAMA PACKAGE
# ============================================================

import os
PKG = Path(os.environ.get("AMD_R22_PACKAGE_ROOT", Path.cwd())).resolve()

assert PKG.exists(), f"Folder package tidak ditemukan: {PKG}"

# ============================================================
# 2. IMPORT SHARED MODULE
# ============================================================

SRC_DIR = PKG / "src"
COMMON_FILE = SRC_DIR / "amd_r22_common.py"

assert COMMON_FILE.exists(), (
    f"Module tidak ditemukan: {COMMON_FILE}\n"
    "Pastikan Anda menggunakan package v1.2 dan struktur folder tidak diubah."
)

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from amd_r22_common import (
    extract_zip_if_needed,
    resolve_manifest_paths,
    validate_manifest,
    set_seed,
    sha256_file,
    validate_predictions,
    seed_level_pooled_metrics,
    metric_record,
)

print("Shared module imported successfully.")

# ============================================================
# 3. LOAD CONFIGURATION
# ============================================================

CONFIG_PATH = (
    PKG
    / "config"
    / "AMD_R2_2_BASELINE_CONFIG_LOCK_v1.yaml"
)

assert CONFIG_PATH.exists(), f"Config tidak ditemukan: {CONFIG_PATH}"

with open(CONFIG_PATH, "r", encoding="utf-8") as file:
    CFG = yaml.safe_load(file)

RCFG = CFG["retfound"]

print("Configuration loaded.")
print("Method ID:", RCFG["method_id"])
print("Expected embedding dimension:", RCFG["embedding_dim_expected"])

# ============================================================
# 4. TEMUKAN ROI_594.zip SECARA OTOMATIS
# ============================================================

roi_candidates = [
    PKG / "ROI_594.zip",
    PKG.parent / "ROI_594.zip",
    Path.cwd() / "ROI_594.zip",
]

ROI_ZIP = next(
    (path for path in roi_candidates if path.exists()),
    None,
)

assert ROI_ZIP is not None, (
    "ROI_594.zip tidak ditemukan.\n"
    "Letakkan file tersebut di folder utama package atau satu folder "
    "di atas package."
)

print("ROI ZIP:", ROI_ZIP)

# ============================================================
# 5. EXTRACT ROI JIKA BELUM DIEKSTRAK
# ============================================================

DATA_DIR = PKG / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

ROI_ROOT = extract_zip_if_needed(
    ROI_ZIP,
    DATA_DIR,
)

print("ROI root:", ROI_ROOT)

# ============================================================
# 6. LOAD CANONICAL 594-IMAGE MANIFEST
# ============================================================

MANIFEST_PATH = (
    PKG
    / "manifests"
    / "AMD_DATASET_594_R2_2_MANIFEST.csv"
)

assert MANIFEST_PATH.exists(), (
    f"Manifest tidak ditemukan: {MANIFEST_PATH}"
)

MANIFEST = pd.read_csv(MANIFEST_PATH)

MANIFEST = resolve_manifest_paths(
    MANIFEST,
    ROI_ROOT,
)

assert len(MANIFEST) == 594, (
    f"Manifest harus berisi 594 baris, ditemukan {len(MANIFEST)}"
)

assert MANIFEST["sample_id"].nunique() == 594, (
    "sample_id dalam manifest tidak unik."
)

print("Manifest loaded:", MANIFEST.shape)
print("\nClass counts:")
print(MANIFEST["true_label"].value_counts().sort_index())

print("\nSource counts:")
print(MANIFEST["source_dataset"].value_counts().sort_index())

print("\nFold counts:")
print(MANIFEST["outer_fold"].value_counts().sort_index())

# ============================================================
# 7. VALIDATE CANONICAL RELEASE
# ============================================================

FOLD_PATH = (
    PKG
    / "manifests"
    / "outer_fold_assignments.csv"
)

assert FOLD_PATH.exists(), (
    f"Outer-fold file tidak ditemukan: {FOLD_PATH}"
)

preflight = validate_manifest(
    MANIFEST,
    ROI_ROOT,
    CFG["project"]["outer_fold_sha256"],
    FOLD_PATH,
    True,
)

assert preflight["complete_and_valid"], preflight

print("\nCanonical preflight passed.")
print("complete_and_valid:", preflight["complete_and_valid"])

# ============================================================
# 8. DEFINE OUTPUT DIRECTORY
# ============================================================

OUTPUT = PKG / "outputs" / "retfound"
OUTPUT.mkdir(
    parents=True,
    exist_ok=True,
)

print("\nOutput directory:", OUTPUT)

# ============================================================
# 9. DEFINE CLASS ORDER FOR LATER CELLS
# ============================================================

LABELS = CFG["project"]["classes"]

print("Canonical labels:", LABELS)
print("\nSetup recovery complete.")

In [ ]:
required_variables = [
    "PKG",
    "CFG",
    "RCFG",
    "MANIFEST",
    "OUTPUT",
    "ROI_ROOT",
    "model",
    "DEVICE",
    "ROIDataset",
]

missing_variables = [
    variable
    for variable in required_variables
    if variable not in globals()
]

print("Missing variables:", missing_variables)

assert not missing_variables, (
    "Masih ada variabel yang belum tersedia: "
    + ", ".join(missing_variables)
)

In [ ]:
required_variables = [
    "RCFG",
    "MANIFEST",
    "model",
    "DEVICE",
    "ROIDataset",
]

missing_variables = [
    name for name in required_variables
    if name not in globals()
]

assert not missing_variables, (
    "Variabel berikut belum tersedia: "
    + ", ".join(missing_variables)
    + ". Jalankan kembali cell setup/model sebelumnya."
)

assert len(MANIFEST) == 594, (
    f"Manifest harus berisi 594 citra, ditemukan {len(MANIFEST)}"
)

assert int(RCFG["embedding_dim_expected"]) == 1024

print("Semua variabel extraction tersedia.")
print("Manifest rows:", len(MANIFEST))
print("Device:", DEVICE)
print("Expected embedding dimension:", RCFG["embedding_dim_expected"])

In [ ]:
import time
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

CACHE_NPZ = OUTPUT / "retfound_mae_cfp_features_594.npz"
CACHE_META = OUTPUT / "retfound_mae_cfp_feature_metadata.csv"

if not CACHE_NPZ.exists():

    loader = DataLoader(
        ROIDataset(MANIFEST),
        batch_size=int(RCFG["batch_size"]),
        shuffle=False,
        num_workers=int(RCFG["num_workers"]),
        pin_memory=torch.cuda.is_available(),
    )

    feature_batches = []
    sample_ids = []

    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

    start_time = time.perf_counter()

    model.eval()

    with torch.inference_mode():
        for batch_index, (images, ids) in enumerate(loader, start=1):

            images = images.to(
                DEVICE,
                non_blocking=torch.cuda.is_available(),
            )

            embeddings = model.forward_features(images)

            # Normalisasi bentuk output RETFound.
            if embeddings.ndim == 3 and embeddings.shape[1] == 1:
                embeddings = embeddings[:, 0, :]

            assert embeddings.ndim == 2, embeddings.shape
            assert embeddings.shape[1] == 1024, embeddings.shape
            assert torch.isfinite(embeddings).all()

            feature_batches.append(
                embeddings.detach().cpu().numpy().astype(np.float32)
            )
            sample_ids.extend([str(x) for x in ids])

            if batch_index % 25 == 0:
                print(
                    f"Processed {len(sample_ids)}/594 images"
                )

    extract_seconds = time.perf_counter() - start_time
    X = np.concatenate(feature_batches, axis=0)

    assert X.shape == (594, 1024), X.shape
    assert len(sample_ids) == 594
    assert sample_ids == MANIFEST["sample_id"].astype(str).tolist()
    assert np.isfinite(X).all()

    np.savez_compressed(
        CACHE_NPZ,
        sample_id=np.asarray(sample_ids),
        features=X,
    )

    MANIFEST[
        [
            "sample_id",
            "source_dataset",
            "true_label",
            "outer_fold",
        ]
    ].to_csv(
        CACHE_META,
        index=False,
    )

else:
    print("Existing feature cache found. Loading it...")

    cache = np.load(
        CACHE_NPZ,
        allow_pickle=False,
    )

    sample_ids = cache["sample_id"].astype(str).tolist()
    X = cache["features"]
    extract_seconds = np.nan

# Final cache audit
assert X.shape == (594, 1024), X.shape
assert sample_ids == MANIFEST["sample_id"].astype(str).tolist()
assert np.isfinite(X).all()

print("Feature shape      :", X.shape)
print("Data type          :", X.dtype)
print("Finite values      :", bool(np.isfinite(X).all()))
print("Extraction seconds :", extract_seconds)
print("Cache saved at     :", CACHE_NPZ)
print("Metadata saved at  :", CACHE_META)

In [ ]:

CACHE_NPZ=OUTPUT/'retfound_mae_cfp_features_594.npz'
CACHE_META=OUTPUT/'retfound_mae_cfp_feature_metadata.csv'
records=[]
if not CACHE_NPZ.exists():
    loader=DataLoader(ROIDataset(MANIFEST),batch_size=int(RCFG['batch_size']),shuffle=False,
                      num_workers=int(RCFG['num_workers']),pin_memory=torch.cuda.is_available())
    feats=[]; ids=[]
    if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()
    t0=time.perf_counter()
    with torch.inference_mode():
        for x,sid in loader:
            z=model.forward_features(x.to(DEVICE,non_blocking=True))
            if z.ndim==3 and z.shape[1]==1: z=z[:,0,:]
            feats.append(z.detach().cpu().numpy().astype(np.float32)); ids.extend(list(sid))
    extract_seconds=time.perf_counter()-t0
    X=np.concatenate(feats,axis=0)
    assert X.shape==(594,int(RCFG['embedding_dim_expected'])),X.shape
    np.savez_compressed(CACHE_NPZ,sample_id=np.array(ids),features=X)
    MANIFEST[['sample_id','source_dataset','true_label','outer_fold']].to_csv(CACHE_META,index=False)
else:
    d=np.load(CACHE_NPZ,allow_pickle=False); ids=d['sample_id'].astype(str).tolist(); X=d['features']; extract_seconds=np.nan
assert ids==MANIFEST.sample_id.tolist()
assert np.isfinite(X).all()
print(X.shape,'extraction seconds:',extract_seconds)


## Five-seed immutable-fold pooled OOF evaluation

In [ ]:

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVC

rows=[]; timing=[]
labels=LABELS
for seed in CFG['project']['seeds']:
    set_seed(seed)
    for fold in CFG['project']['outer_folds']:
        tr=MANIFEST.outer_fold.ne(fold).to_numpy(); te=~tr
        pipe=Pipeline([
            ('smote',SMOTE(k_neighbors=5,random_state=seed)),
            ('scaler',StandardScaler()),
            ('pca',PCA(n_components=128,random_state=seed)),
            ('svm',SVC(C=10,gamma='scale',kernel='rbf',class_weight='balanced',probability=True,random_state=seed)),
        ])
        t0=time.perf_counter(); pipe.fit(X[tr],MANIFEST.true_label.to_numpy()[tr]); fit_s=time.perf_counter()-t0
        t0=time.perf_counter(); proba=pipe.predict_proba(X[te]); pred=pipe.classes_[np.argmax(proba,axis=1)]; pred_s=time.perf_counter()-t0
        test_df=MANIFEST.loc[te].reset_index(drop=True)
        class_to_col={c:i for i,c in enumerate(pipe.classes_)}
        for i,r in test_df.iterrows():
            rows.append({'method':RCFG['method_id'],'seed':seed,'outer_fold':fold,'sample_id':r.sample_id,
                         'source_dataset':r.source_dataset,'true_label':r.true_label,'predicted_label':pred[i],
                         'correct':int(pred[i]==r.true_label),
                         **{f'prob_{c}':float(proba[i,class_to_col[c]]) for c in labels}})
        timing.append({'method':RCFG['method_id'],'scope':'cached_feature_outer_fold','seed':seed,'outer_fold':fold,
                       'fit_seconds':fit_s,'prediction_seconds':pred_s,'n_train':int(tr.sum()),'n_test':int(te.sum())})
pred_df=pd.DataFrame(rows)
audit=validate_predictions(pred_df,expected_rows=2970)
assert audit['complete_and_valid'],audit
metric_df=seed_level_pooled_metrics(pred_df)
pred_df.to_csv(OUTPUT/'retfound_internal_sample_predictions.csv',index=False)
metric_df.to_csv(OUTPUT/'retfound_internal_seed_level_pooled_metrics.csv',index=False)
pd.DataFrame(timing).to_csv(OUTPUT/'retfound_internal_computational_records.csv',index=False)
print(metric_df)


## LODO cross-repository evaluation

In [ ]:

lodo_rows=[]; lodo_timing=[]
for seed in CFG['project']['seeds']:
    set_seed(seed)
    for held in CFG['project']['sources']:
        tr=MANIFEST.source_dataset.ne(held).to_numpy(); te=~tr
        pipe=Pipeline([
            ('smote',SMOTE(k_neighbors=5,random_state=seed)),('scaler',StandardScaler()),
            ('pca',PCA(n_components=128,random_state=seed)),
            ('svm',SVC(C=10,gamma='scale',kernel='rbf',class_weight='balanced',probability=True,random_state=seed)),])
        t0=time.perf_counter(); pipe.fit(X[tr],MANIFEST.true_label.to_numpy()[tr]); fit_s=time.perf_counter()-t0
        t0=time.perf_counter(); proba=pipe.predict_proba(X[te]); pred=pipe.classes_[np.argmax(proba,axis=1)]; pred_s=time.perf_counter()-t0
        test_df=MANIFEST.loc[te].reset_index(drop=True); class_to_col={c:i for i,c in enumerate(pipe.classes_)}
        for i,r in test_df.iterrows():
            lodo_rows.append({'method':RCFG['method_id'],'seed':seed,'held_out_source':held,'sample_id':r.sample_id,
                              'source_dataset':r.source_dataset,'true_label':r.true_label,'predicted_label':pred[i],
                              'correct':int(pred[i]==r.true_label),**{f'prob_{c}':float(proba[i,class_to_col[c]]) for c in labels}})
        lodo_timing.append({'method':RCFG['method_id'],'scope':'cached_feature_lodo','seed':seed,'held_out_source':held,
                            'fit_seconds':fit_s,'prediction_seconds':pred_s,'n_train':int(tr.sum()),'n_test':int(te.sum())})
lodo=pd.DataFrame(lodo_rows)
assert len(lodo)==2970 and not lodo.duplicated(['method','seed','held_out_source','sample_id']).any()
lodo.to_csv(OUTPUT/'retfound_lodo_sample_predictions.csv',index=False)
pd.DataFrame(lodo_timing).to_csv(OUTPUT/'retfound_lodo_computational_records.csv',index=False)
summary=[]
for (seed,held),g in lodo.groupby(['seed','held_out_source']): summary.append({'seed':seed,'held_out_source':held,'n':len(g),**metric_record(g.true_label,g.predicted_label)})
pd.DataFrame(summary).to_csv(OUTPUT/'retfound_lodo_metrics.csv',index=False)


## Final machine-readable audit

In [ ]:

final_audit={
 'method':RCFG['method_id'],'complete_and_valid':True,'release_preflight':preflight,
 'internal_prediction_audit':audit,'checkpoint_sha256':sha256_file(CKPT),
 'official_repo_commit':commit,'feature_shape':list(X.shape),'feature_cache_sha256':sha256_file(CACHE_NPZ),
 'metric_aggregation':'seed-level pooled OOF mean ± SD; fixed seed-42 pooled OOF separately; consensus only as paired diagnostic',
 'software':{'python':platform.python_version(),'torch':torch.__version__},
}
(OUTPUT/'RETFOUND_R2_2_VALIDATION_AUDIT.json').write_text(json.dumps(final_audit,indent=2),encoding='utf-8')
print(json.dumps(final_audit,indent=2))
